# Meeting ASR trên Colab GPU
Chọn runtime GPU. Đặt code đã commit và audio vào Drive; không đưa `.env` hay token vào notebook. Kết quả sau đây chỉ có ý nghĩa với audio bạn cung cấp. Xem `TESTING.md` để chạy nghiên cứu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/meeting_asr')
assert (PROJECT_DIR / 'pyproject.toml').is_file(), 'Đặt repo vào PROJECT_DIR trước'
%pip install -q uv

In [ ]:
import os, getpass, subprocess
os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token (đã chấp nhận điều kiện model): ')
os.environ['MEETING_DEVICE'] = 'cuda'
os.environ['MEETING_ASR_BATCH_SIZE'] = '1'
os.environ['MEETING_LLM_ENABLED'] = 'false'
subprocess.run(['uv', 'sync', '--locked', '--extra', 'inference', '--extra', 'api', '--extra', 'dev'], cwd=PROJECT_DIR, check=True)
subprocess.run(['uv', 'run', '--no-sync', 'meeting-asr', 'doctor'], cwd=PROJECT_DIR, check=True)

In [ ]:
AUDIO = PROJECT_DIR / 'data/real/meeting.wav'
OUTPUT = PROJECT_DIR / 'results/colab-demo'
assert AUDIO.is_file(), 'Đặt file audio cần test vào AUDIO'
subprocess.run(['uv', 'run', '--no-sync', 'meeting-asr', 'run', str(AUDIO), '--out', str(OUTPUT)], cwd=PROJECT_DIR, check=True)

In [ ]:
subprocess.run(['uv', 'run', '--no-sync', 'meeting-asr', 'benchmark', str(AUDIO), '--out', str(PROJECT_DIR / 'results/colab-benchmark')], cwd=PROJECT_DIR, check=True)